# V. LFwC: A New Corpus to Demonstrate the Practicability of the Proposed Requirements

We created a Linux Firmware Corpus (LFwC) to assess the practicability of our requirements. It is based on data until June 2023 and consists of 10,913 deduplicated and unpacked firmware images from ten known manufacturers. It includes both actual and historical firmware samples, covering 2,365
unique devices across 22 classes. To provide an overview of LFwC, we added corpus data points to the bottom Table II. We share as much data as legally possible and publish all scripts, tools, and virtual machines for replicability. We tear down LFwC’s unpacking barrier with an open source process
for verified unpacking success.

## Preparations

Below you will find preparatory stuff such as imports and constant definitions for use down the road.

### Imports

In [ ]:
import json
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import rc
from matplotlib.ticker import ScalarFormatter
from packaging.version import Version, parse

### Constants

In [ ]:
CMAP: list[int] = deque(sns.color_palette("colorblind", as_cmap=True))
CMAP.rotate(-4)
CMAP = list(CMAP)

CMAP_2 = deque(CMAP.copy())
CMAP_2.rotate(1)
CMAP_2 = list(CMAP_2)

CORPUS_PATH: Path = Path("../../public_data/lfwc-v2.0-masked.csv")
CORPUS_V1_PATH: Path = Path("../../public_data/lfwc-masked.csv")
CORPUS_V2_FAILED: Path = Path("../../public_data/lfwc-failed-v2.0-masked.csv")
FIGURE_DEST: Path = Path("../../figures/v2")
FIGURE_DEST.mkdir(exist_ok=True)

Y_LABELS: list[str] = [
    "Ubiquiti",
    "TRENDnet",
    "NETGEAR",
    "Linksys",
    "EnGenius",
    "EDIMAX",
    "D-Link",
    "ASUS",
    "TP-Link",
    "FRITZ!",
    "DrayTek",
    "MikroTik",
    "Moxa",
    "Netis",
    "TOTOLINK",
    "Zyxel"
]
Y_LABELS.sort()

### Matplotlib Settings

In [ ]:
rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
rc("text", usetex=True)
pd.set_option("display.max_colwidth", None)

### Read Data

In [ ]:
assert CORPUS_PATH.is_file(), f"Error: {CORPUS_PATH.absolute()=} not found"
df = pd.read_csv(CORPUS_PATH, index_col=0)
df_v1 = pd.read_csv(CORPUS_V1_PATH, index_col=0)
df_v2_failed = pd.read_csv(CORPUS_V2_FAILED, index_col=0)

In [ ]:
def firmware_distribution_per_lfwc_version(df: pd.DataFrame) -> None:
    df_removed_day_from_date = df.copy()
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 16})
    df_removed_day_from_date["release_date"] = (
        df_removed_day_from_date["release_date"].str.split("-").str[:-2].str.join("-")
    )
    df_history = (
        df_removed_day_from_date.groupby(["release_date", "corpus_version"], as_index=False)
        .nunique()
        .pivot(index="release_date", columns=["corpus_version"], values="md5")
        .fillna(value=0.0)
    )
    ax = df_history.plot(
        kind="bar",
        grid=True,
        stacked=True,
        logy=False,
        figsize=(8, 6),
        rot=50,
        legend=False,
        edgecolor="black",
        color=[CMAP[6], CMAP[7]],
    )
    ax.set_xticklabels(["unk."] + [str(i) for i in range(2004, 2026)])
    ax.set_ylabel("Samples")
    ax.set_xlabel("Release Year")
    ax.legend(ncols=4, bbox_to_anchor=(0.92, 1.1), labels=["LFwC v1.0: Included \\& Available", "LFwC v2.0: Added"], fontsize=13)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    plt.tight_layout()
    rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
    plt.savefig(FIGURE_DEST / "release_date_per_lfwc_version.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# create_figure_7_firmware_distribution_per_release_date(df)
firmware_distribution_per_lfwc_version(df)

In [ ]:
def normalize_manufacturer(df):
    df = df.copy()
    df['manufacturer'] = df['manufacturer'].str.lower()
    # AVM und fritz! zu avm/fritz! zusammenfassen
    df['manufacturer'] = df['manufacturer'].replace({
        'avm': 'avm/fritz!',
        'fritz!': 'avm/fritz!',
        'dlink': 'd-link',
    })
    df['sha256'] = df['sha256'].str.lower()
    return df

def create_table_X_v1_vs_v2_distribution_by_vendor_v2(
    f_v1: pd.DataFrame, 
    df_v2: pd.DataFrame,
    df_v2_failed: pd.DataFrame,
) -> None:
    result_v1 = normalize_manufacturer(df_v1).groupby('manufacturer').agg(
        **{
            'Samples v1':
            ('device_name', 'size'),
        }
    ).reset_index()
    
    result_v2 = normalize_manufacturer(df_v2).groupby('manufacturer').agg(
        **{
            'Samples v2': ('device_name', 'size'),
            'FS sum v2': ('unix_fs_structures', 'sum'),
            'FS mean v2': ('unix_fs_structures', 'mean'),
        }
    ).reset_index()
    
    failed_v2 = normalize_manufacturer(df_v2_failed).groupby('manufacturer').agg(
        **{
            'Samples v2 failed': ('device_name', 'size'),
        }
    ).reset_index()

    result = result_v1.merge(result_v2, on='manufacturer', how='outer').fillna(0)
    result = result.merge(failed_v2, on='manufacturer', how='outer').fillna(0)
    numeric_cols = result.select_dtypes(include=['float64']).columns
    #result[numeric_cols] = result[numeric_cols].astype(int)
    result.rename(columns={'manufacturer': 'Vendor'}, inplace=True)
    result['Delta Samples'] = result['Samples v2'] - result['Samples v1']
    result['Scraped'] = result['Samples v2'] + result['Samples v2 failed']

    result.rename(columns={"Samples v2 failed": "Failed", "Samples v2": "Unpacked", "Delta Samples": "Delta v1.0"}, inplace=True)

    result = result[['Vendor', 'Scraped', 'Failed', 'Unpacked', 'Delta v1.0', 'FS sum v2', 'FS mean v2']]#, 'Devices v2', 'Delta Devices']] #'FS sum v2', 'FS mean v2', 'Samples v1', 'Devices v1',

    #print(result["FS sum v2"].sum())
    
    with pd.option_context('display.width', 300):
        print(result)

create_table_X_v1_vs_v2_distribution_by_vendor_v2(df_v1, df, df_v2_failed)